<a href="https://colab.research.google.com/github/SyuzannaKhalatyaning/CS156-Assignment/blob/main/CS156_First_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1: install HEIC support (only needed if you have .heic files) and mount Drive
!pip install -q pillow-heif
from google.colab import drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 29.5 MB/s eta 0:00:00
Mounted at /content/drive


In [3]:
# Cell 2: find all images and count per class

from pathlib import Path
import pandas as pd

DATA_DIR = Path('/content/drive/MyDrive/data')

# folder name on Drive -> short label used in the model and report
FOLDERS = {
    'Work':        'work_study',
    'Socials':     'social',
    'Exploration': 'exploration',
}
EXTS = {'.jpg', '.jpeg', '.png', '.heic'}

rows = []
for folder, label in FOLDERS.items():
    for p in (DATA_DIR / folder).rglob('*'):   # rglob also finds photos in subfolders
        if p.suffix.lower() in EXTS:
            rows.append({'path': str(p), 'label': label})

df = pd.DataFrame(rows)
print(df['label'].value_counts())

label
exploration    230
work_study     188
social          71
Name: count, dtype: int64


In [5]:
for p in DATA_DIR.iterdir():
    print(repr(p.name), '| is_dir:', p.is_dir())

print()
for folder in FOLDERS:
    print(repr(folder), '-> exists:', (DATA_DIR / folder).exists())

'Socials' | is_dir: True
'Exploration' | is_dir: True
'Work' | is_dir: True
'labels_v1.csv' | is_dir: False

'Work' -> exists: True
'Socials' -> exists: True
'Exploration' -> exists: True


In [6]:
import re
from PIL import Image
import pillow_heif
pillow_heif.register_heif_opener()

def get_date(path):
    # 1) try EXIF
    try:
        exif = Image.open(path).getexif()
        d = exif.get_ifd(0x8769).get(36867) or exif.get(306)
        if d:
            return pd.to_datetime(d, format='%Y:%m:%d %H:%M:%S')
    except Exception:
        pass
    # 2) fall back to a date in the filename, e.g. 20260911_152838
    m = re.search(r'(20\d{6})[_-]?(\d{6})?', Path(path).name)
    if m:
        try:
            return pd.to_datetime(m.group(1), format='%Y%m%d')
        except Exception:
            pass
    return pd.NaT

df['date'] = df['path'].apply(get_date)
print(df['date'].isna().sum(), 'photos have no date')
print(df['date'].min(), '->', df['date'].max())
df = df.sort_values('date').reset_index(drop=True)

0 photos have no date
2024-08-07 13:25:33 -> 2026-10-12 21:51:16


In [8]:
df.to_csv('/content/drive/MyDrive/data/labels_v1.csv', index=False)